# Marketplace Analytics Platform
## Notebook 4 - Warehouse Validation 

This notebook validates the warehouse after the ETL pipeline has run. It checks five things in order:

1. **Connection** — confirm Python can reach the database
2. **Row counts** — verify every table loaded the expected number of rows
3. **Referential integrity** — confirm all foreign key relationships are intact
4. **Null checks** — verify NOT NULL columns contain no nulls
5. **Business logic** — verify cleaning rules were applied correctly

---
**Run order:** Run `01_load_staging.py` and `02_load_warehouse.py` before this notebook.  
**Expected outcome:** Every check prints ✓ PASS. Any ✗ FAIL requires investigation before moving to next Phase.

## Setup

In [ ]:
import sys
from pathlib import Path

import pandas as pd
from sqlalchemy import text

# Add utils to path (notebook is in notebooks/, project root is one level up)
sys.path.insert(0, str(Path.cwd().parents[0] / 'python' / 'utils'))
from db_connection import get_engine

engine = get_engine()

# Helper: run a query and return a DataFrame
def q(sql):
    with engine.connect() as conn:
        return pd.read_sql(text(sql), conn)

# Helper: print pass/fail
def check(label, passed, detail=''):
    status = '✓ PASS' if passed else '✗ FAIL'
    suffix = f'  → {detail}' if detail else ''
    print(f'{status}  {label}{suffix}')

print('Connected to:', q('SELECT current_database()').iloc[0, 0])

Connected to: Marketplace-Analytics-Platform


---
## 1. Row Counts

Every warehouse table should contain exactly the expected number of rows after a clean ETL run. These counts are derived from the source CSV row counts, accounting for deduplication in `dim_geolocation`.

In [2]:
EXPECTED_COUNTS = {
    'dim_date':         (1461,    1461),     # 2016-01-01 to 2019-12-31
    'dim_geolocation':  (18_000,  22_000),   # deduplicated from 1,000,163 raw rows
    'dim_customer':     (99_441,  99_441),
    'dim_seller':       (3_095,   3_095),
    'dim_product':      (32_951,  32_951),
    'dim_order':        (99_441,  99_441),
    'fact_order_items': (112_650, 112_650),
    'fact_payments':    (103_886, 103_886),
    'fact_reviews':     (99_224,  99_224),
}

print('Row Count Validation')
print('─' * 55)
all_pass = True
for table, (lo, hi) in EXPECTED_COUNTS.items():
    count = q(f'SELECT COUNT(*) AS n FROM warehouse."{table}"').iloc[0]['n']
    passed = lo <= count <= hi
    if not passed:
        all_pass = False
    check(f'{table:<25} {count:>9,} rows', passed,
          '' if passed else f'expected [{lo:,} – {hi:,}]')

print('─' * 55)
print('All row count checks passed ✓' if all_pass else '⚠ One or more row count checks FAILED')

Row Count Validation
───────────────────────────────────────────────────────
✓ PASS  dim_date                      1,461 rows
✓ PASS  dim_geolocation              19,011 rows
✓ PASS  dim_customer                 99,441 rows
✓ PASS  dim_seller                    3,095 rows
✓ PASS  dim_product                  32,951 rows
✓ PASS  dim_order                    99,441 rows
✓ PASS  fact_order_items            112,650 rows
✓ PASS  fact_payments               103,886 rows
✓ PASS  fact_reviews                 99,224 rows
───────────────────────────────────────────────────────
All row count checks passed ✓


---
## 2. Referential Integrity

Although Postgres FK constraints enforce most relationships at the DDL level, `fact_payments` and `fact_reviews` join via `order_id` (no FK constraint). This section manually verifies that all FK relationships resolve correctly — no orphaned keys.

A count of 0 for each check means no orphaned records exist.

In [3]:
print('Referential Integrity Checks')
print('─' * 55)

ri_checks = {
    'dim_order → dim_customer': """
        SELECT COUNT(*) AS n FROM warehouse.dim_order o
        LEFT JOIN warehouse.dim_customer c ON o.customer_key = c.customer_key
        WHERE c.customer_key IS NULL
    """,
    'dim_order → dim_date (purchase_date_key)': """
        SELECT COUNT(*) AS n FROM warehouse.dim_order o
        LEFT JOIN warehouse.dim_date d ON o.purchase_date_key = d.date_key
        WHERE o.purchase_date_key IS NOT NULL AND d.date_key IS NULL
    """,
    'fact_order_items → dim_order': """
        SELECT COUNT(*) AS n FROM warehouse.fact_order_items f
        LEFT JOIN warehouse.dim_order o ON f.order_key = o.order_key
        WHERE o.order_key IS NULL
    """,
    'fact_order_items → dim_seller': """
        SELECT COUNT(*) AS n FROM warehouse.fact_order_items f
        LEFT JOIN warehouse.dim_seller s ON f.seller_key = s.seller_key
        WHERE s.seller_key IS NULL
    """,
    'fact_order_items → dim_product': """
        SELECT COUNT(*) AS n FROM warehouse.fact_order_items f
        LEFT JOIN warehouse.dim_product p ON f.product_key = p.product_key
        WHERE p.product_key IS NULL
    """,
    'fact_payments → dim_order (via order_id)': """
        SELECT COUNT(*) AS n FROM warehouse.fact_payments fp
        LEFT JOIN warehouse.dim_order o ON fp.order_id = o.order_id
        WHERE o.order_id IS NULL
    """,
    'fact_reviews → dim_order (via order_id)': """
        SELECT COUNT(*) AS n FROM warehouse.fact_reviews fr
        LEFT JOIN warehouse.dim_order o ON fr.order_id = o.order_id
        WHERE o.order_id IS NULL
    """,
}

all_pass = True
for label, sql in ri_checks.items():
    orphans = q(sql).iloc[0]['n']
    passed  = orphans == 0
    if not passed:
        all_pass = False
    check(label, passed, '' if passed else f'{orphans:,} orphaned rows')

print('─' * 55)
print('All referential integrity checks passed ✓' if all_pass else '⚠ One or more RI checks FAILED')

Referential Integrity Checks
───────────────────────────────────────────────────────
✓ PASS  dim_order → dim_customer
✓ PASS  dim_order → dim_date (purchase_date_key)
✓ PASS  fact_order_items → dim_order
✓ PASS  fact_order_items → dim_seller
✓ PASS  fact_order_items → dim_product
✓ PASS  fact_payments → dim_order (via order_id)
✓ PASS  fact_reviews → dim_order (via order_id)
───────────────────────────────────────────────────────
All referential integrity checks passed ✓


---
## 3. Null Checks on NOT NULL Columns

Columns defined as `NOT NULL` in the DDL should contain zero null values. This confirms the ETL did not silently drop or corrupt required fields.

In [4]:
print('NOT NULL Column Checks')
print('─' * 55)

not_null_checks = {
    'dim_date.date_key':                      'SELECT COUNT(*) AS n FROM warehouse.dim_date WHERE date_key IS NULL',
    'dim_date.full_date':                     'SELECT COUNT(*) AS n FROM warehouse.dim_date WHERE full_date IS NULL',
    'dim_geolocation.zip_code_prefix':        'SELECT COUNT(*) AS n FROM warehouse.dim_geolocation WHERE zip_code_prefix IS NULL',
    'dim_geolocation.avg_lat':                'SELECT COUNT(*) AS n FROM warehouse.dim_geolocation WHERE avg_lat IS NULL',
    'dim_geolocation.avg_lng':                'SELECT COUNT(*) AS n FROM warehouse.dim_geolocation WHERE avg_lng IS NULL',
    'dim_customer.customer_id':               'SELECT COUNT(*) AS n FROM warehouse.dim_customer WHERE customer_id IS NULL',
    'dim_customer.customer_unique_id':        'SELECT COUNT(*) AS n FROM warehouse.dim_customer WHERE customer_unique_id IS NULL',
    'dim_seller.seller_id':                   'SELECT COUNT(*) AS n FROM warehouse.dim_seller WHERE seller_id IS NULL',
    'dim_product.product_id':                 'SELECT COUNT(*) AS n FROM warehouse.dim_product WHERE product_id IS NULL',
    'dim_order.order_id':                     'SELECT COUNT(*) AS n FROM warehouse.dim_order WHERE order_id IS NULL',
    'dim_order.order_status':                 'SELECT COUNT(*) AS n FROM warehouse.dim_order WHERE order_status IS NULL',
    'dim_order.order_purchase_timestamp':     'SELECT COUNT(*) AS n FROM warehouse.dim_order WHERE order_purchase_timestamp IS NULL',
    'fact_order_items.order_id':              'SELECT COUNT(*) AS n FROM warehouse.fact_order_items WHERE order_id IS NULL',
    'fact_order_items.price':                 'SELECT COUNT(*) AS n FROM warehouse.fact_order_items WHERE price IS NULL',
    'fact_order_items.freight_value':         'SELECT COUNT(*) AS n FROM warehouse.fact_order_items WHERE freight_value IS NULL',
    'fact_payments.order_id':                 'SELECT COUNT(*) AS n FROM warehouse.fact_payments WHERE order_id IS NULL',
    'fact_payments.payment_type':             'SELECT COUNT(*) AS n FROM warehouse.fact_payments WHERE payment_type IS NULL',
    'fact_payments.payment_installments':     'SELECT COUNT(*) AS n FROM warehouse.fact_payments WHERE payment_installments IS NULL',
    'fact_payments.payment_value':            'SELECT COUNT(*) AS n FROM warehouse.fact_payments WHERE payment_value IS NULL',
    'fact_reviews.review_id':                 'SELECT COUNT(*) AS n FROM warehouse.fact_reviews WHERE review_id IS NULL',
    'fact_reviews.order_id':                  'SELECT COUNT(*) AS n FROM warehouse.fact_reviews WHERE order_id IS NULL',
    'fact_reviews.review_score':              'SELECT COUNT(*) AS n FROM warehouse.fact_reviews WHERE review_score IS NULL',
}

all_pass = True
for label, sql in not_null_checks.items():
    nulls  = q(sql).iloc[0]['n']
    passed = nulls == 0
    if not passed:
        all_pass = False
    check(f'{label:<45}', passed, '' if passed else f'{nulls:,} null values found')

print('─' * 55)
print('All null checks passed ✓' if all_pass else '⚠ One or more null checks FAILED')

NOT NULL Column Checks
───────────────────────────────────────────────────────
✓ PASS  dim_date.date_key                            
✓ PASS  dim_date.full_date                           
✓ PASS  dim_geolocation.zip_code_prefix              
✓ PASS  dim_geolocation.avg_lat                      
✓ PASS  dim_geolocation.avg_lng                      
✓ PASS  dim_customer.customer_id                     
✓ PASS  dim_customer.customer_unique_id              
✓ PASS  dim_seller.seller_id                         
✓ PASS  dim_product.product_id                       
✓ PASS  dim_order.order_id                           
✓ PASS  dim_order.order_status                       
✓ PASS  dim_order.order_purchase_timestamp           
✓ PASS  fact_order_items.order_id                    
✓ PASS  fact_order_items.price                       
✓ PASS  fact_order_items.freight_value               
✓ PASS  fact_payments.order_id                       
✓ PASS  fact_payments.payment_type                   
✓ P

---
## 4. Business Logic Checks

These checks verify that the specific cleaning rules documented in `05_etl_methodology.md` were correctly applied. Each check targets one documented decision.

In [5]:
print('Business Logic — Cleaning Rule Verification')
print('─' * 60)

# --- Rule: product_weight_g = 0 must be NULL ---
n = q("SELECT COUNT(*) AS n FROM warehouse.dim_product WHERE product_weight_g = 0").iloc[0]['n']
check('No product_weight_g = 0 rows (should be NULL)', n == 0,
      '' if n == 0 else f'{n} rows still have weight = 0')

# --- Rule: payment_installments must be >= 1 ---
n = q("SELECT COUNT(*) AS n FROM warehouse.fact_payments WHERE payment_installments < 1").iloc[0]['n']
check('No payment_installments < 1', n == 0,
      '' if n == 0 else f'{n} rows have installments < 1')

# --- Rule: shipping_limit_date outliers nulled out ---
n = q("""
    SELECT COUNT(*) AS n
    FROM warehouse.fact_order_items f
    JOIN warehouse.dim_order o ON f.order_key = o.order_key
    WHERE f.shipping_limit_date > o.order_purchase_timestamp + INTERVAL '60 days'
""").iloc[0]['n']
check('No shipping_limit_date > purchase + 60 days', n == 0,
      '' if n == 0 else f'{n} outlier rows remain')

# --- Rule: review_score must be between 1 and 5 ---
n = q("SELECT COUNT(*) AS n FROM warehouse.fact_reviews WHERE review_score NOT BETWEEN 1 AND 5").iloc[0]['n']
check('All review_score values between 1 and 5', n == 0,
      '' if n == 0 else f'{n} out-of-range scores')

# --- Rule: price and freight_value must be > 0 ---
n = q("SELECT COUNT(*) AS n FROM warehouse.fact_order_items WHERE price <= 0").iloc[0]['n']
check('No price <= 0 in fact_order_items', n == 0,
      '' if n == 0 else f'{n} rows with price <= 0')

n = q("SELECT COUNT(*) AS n FROM warehouse.fact_order_items WHERE freight_value < 0").iloc[0]['n']
check('No freight_value < 0 in fact_order_items', n == 0,
      '' if n == 0 else f'{n} rows with negative freight')

# --- Rule: order_purchase_timestamp within expected date range ---
n = q("""
    SELECT COUNT(*) AS n FROM warehouse.dim_order
    WHERE order_purchase_timestamp < '2016-01-01'
       OR order_purchase_timestamp > '2018-12-31'
""").iloc[0]['n']
check('All purchase timestamps within 2016–2018', n == 0,
      '' if n == 0 else f'{n} orders outside expected date range')

# --- Rule: dim_date covers 2016-01-01 to 2019-12-31 ---
bounds = q("SELECT MIN(full_date) AS min_d, MAX(full_date) AS max_d FROM warehouse.dim_date").iloc[0]
passed = str(bounds['min_d']) == '2016-01-01' and str(bounds['max_d']) == '2019-12-31'
check('dim_date spans 2016-01-01 → 2019-12-31', passed,
      f'actual range: {bounds["min_d"]} → {bounds["max_d"]}')

print('─' * 60)

Business Logic — Cleaning Rule Verification
────────────────────────────────────────────────────────────
✓ PASS  No product_weight_g = 0 rows (should be NULL)
✓ PASS  No payment_installments < 1
✓ PASS  No shipping_limit_date > purchase + 60 days
✓ PASS  All review_score values between 1 and 5
✓ PASS  No price <= 0 in fact_order_items
✓ PASS  No freight_value < 0 in fact_order_items
✓ PASS  All purchase timestamps within 2016–2018
✓ PASS  dim_date spans 2016-01-01 → 2019-12-31  → actual range: 2016-01-01 → 2019-12-31
────────────────────────────────────────────────────────────


---
## 5. Spot Checks — Sample Data Review

A final visual inspection of sample rows from key tables. This catches issues that automated checks miss — unexpected values, encoding problems, or odd formatting in string columns.

In [6]:
print('── dim_date (first 5 rows) ──')
display(q('SELECT * FROM warehouse.dim_date ORDER BY date_key LIMIT 5'))

── dim_date (first 5 rows) ──


,date_key,full_date,day,month,month_name,quarter,year,week_of_year,weekday_name,is_weekend
0,20160101,2016-01-01,1,1,January,1,2016,53,Friday,False
1,20160102,2016-01-02,2,1,January,1,2016,53,Saturday,True
2,20160103,2016-01-03,3,1,January,1,2016,53,Sunday,True
3,20160104,2016-01-04,4,1,January,1,2016,1,Monday,False
4,20160105,2016-01-05,5,1,January,1,2016,1,Tuesday,False


In [7]:
print('── dim_geolocation (5 sample rows) ──')
display(q('SELECT * FROM warehouse.dim_geolocation LIMIT 5'))

── dim_geolocation (5 sample rows) ──


,geolocation_key,zip_code_prefix,avg_lat,avg_lng,city,state
0,1,01001,-23.550190,-46.634023,sao paulo,SP
1,2,01002,-23.548146,-46.634979,sao paulo,SP
2,3,01003,-23.548994,-46.635731,sao paulo,SP
3,4,01004,-23.549799,-46.634757,sao paulo,SP
4,5,01005,-23.549456,-46.636733,sao paulo,SP


In [8]:
print('── dim_customer (5 sample rows) ──')
display(q('SELECT * FROM warehouse.dim_customer LIMIT 5'))

── dim_customer (5 sample rows) ──


,customer_key,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,geolocation_key
0,1,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP,5359
1,2,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,09790,sao bernardo do campo,SP,4280
2,3,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,01151,sao paulo,SP,85
3,4,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,08775,mogi das cruzes,SP,4029
4,5,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP,4836


In [9]:
print('── dim_product — categories with NULL English name (source blanks) ──')
display(q("""
    SELECT product_category_name, COUNT(*) AS product_count
    FROM warehouse.dim_product
    WHERE product_category_name_english IS NULL
    GROUP BY product_category_name
    ORDER BY product_count DESC
    LIMIT 10
"""))

── dim_product — categories with NULL English name (source blanks) ──


,product_category_name,product_count
0,None,610


In [10]:
print('── dim_order — order status distribution ──')
display(q("""
    SELECT order_status, COUNT(*) AS order_count,
           ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS pct
    FROM warehouse.dim_order
    GROUP BY order_status
    ORDER BY order_count DESC
"""))

── dim_order — order status distribution ──


,order_status,order_count,pct
0,delivered,96478,97.02
1,shipped,1107,1.11
2,canceled,625,0.63
3,unavailable,609,0.61
4,invoiced,314,0.32
5,processing,301,0.30
6,created,5,0.01
7,approved,2,0.00


In [11]:
print('── fact_order_items — price and freight summary ──')
display(q("""
    SELECT
        COUNT(*)                        AS total_items,
        ROUND(MIN(price), 2)            AS min_price,
        ROUND(AVG(price), 2)            AS avg_price,
        ROUND(MAX(price), 2)            AS max_price,
        ROUND(MIN(freight_value), 2)    AS min_freight,
        ROUND(AVG(freight_value), 2)    AS avg_freight,
        ROUND(MAX(freight_value), 2)    AS max_freight,
        COUNT(shipping_limit_date)      AS non_null_shipping_dates
    FROM warehouse.fact_order_items
"""))

── fact_order_items — price and freight summary ──


,total_items,min_price,avg_price,max_price,min_freight,avg_freight,max_freight,non_null_shipping_dates
0,112650,0.85,120.65,6735.0,0.0,19.99,409.68,112638


In [12]:
print('── fact_payments — payment type distribution ──')
display(q("""
    SELECT payment_type,
           COUNT(*)                                             AS payment_count,
           ROUND(AVG(payment_installments), 2)                 AS avg_installments,
           ROUND(AVG(payment_value), 2)                        AS avg_value,
           ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS pct
    FROM warehouse.fact_payments
    GROUP BY payment_type
    ORDER BY payment_count DESC
"""))

── fact_payments — payment type distribution ──


,payment_type,payment_count,avg_installments,avg_value,pct
0,credit_card,76795,3.51,163.32,73.92
1,boleto,19784,1.00,145.03,19.04
2,voucher,5775,1.00,65.70,5.56
3,debit_card,1529,1.00,142.57,1.47
4,not_defined,3,1.00,0.00,0.00


In [13]:
print('── fact_reviews — review score distribution ──')
display(q("""
    SELECT review_score,
           COUNT(*)                                             AS review_count,
           ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS pct
    FROM warehouse.fact_reviews
    GROUP BY review_score
    ORDER BY review_score DESC
"""))

── fact_reviews — review score distribution ──


,review_score,review_count,pct
0,5,57328,57.78
1,4,19142,19.29
2,3,8179,8.24
3,2,3151,3.18
4,1,11424,11.51


---
## Validation Summary

Re-run this cell last to get a final confirmation across all checks.

In [14]:
print('=' * 60)
print('WAREHOUSE VALIDATION — FINAL SUMMARY')
print('=' * 60)
sections = [
    'Row Counts             — run Section 1 to confirm',
    'Referential Integrity  — run Section 2 to confirm',
    'NOT NULL Columns       — run Section 3 to confirm',
    'Business Logic Rules   — run Section 4 to confirm',
    'Spot Checks            — reviewed visually in Section 5',
]
for s in sections:
    print(f'  • {s}')
print('=' * 60)
print('If all checks above passed, the warehouse is ready for Next Phase — SQL Analytical Views.')

WAREHOUSE VALIDATION — FINAL SUMMARY
  • Row Counts             — run Section 1 to confirm
  • Referential Integrity  — run Section 2 to confirm
  • NOT NULL Columns       — run Section 3 to confirm
  • Business Logic Rules   — run Section 4 to confirm
  • Spot Checks            — reviewed visually in Section 5
If all checks above passed, the warehouse is ready for Next Phase — SQL Analytical Views.
